# TP 2.3 – Classification automatique de tweets de catastrophe
## Partie 3 : Implémentation des techniques NLP, des modèles ML/DL et évaluation des performances

**Cas d'usage.** Lors d'une catastrophe (incendie, séisme, inondation, attentat…), Twitter/X est souvent la première source d'information : les témoins publient avant même l'arrivée des médias. Les services de secours, les ONG et les agences de presse ont donc intérêt à **filtrer automatiquement** les tweets qui annoncent une *vraie* catastrophe.

La difficulté est linguistique : le vocabulaire des catastrophes est très souvent utilisé au sens figuré. *"The sky was ablaze last night"* (métaphore) et *"Forest fire near La Ronge"* (vraie catastrophe) partagent le même champ lexical. Un simple filtre par mots-clés ne suffit pas : il faut un modèle qui apprenne le **contexte**.

**Jeu de données.** *NLP with Disaster Tweets – cleaning data* (Kaggle, auteur : vbmokin). C'est une version nettoyée du jeu de la compétition Kaggle *"Real or Not? NLP with Disaster Tweets"* : les URLs sont retirées, les contractions développées, les fautes corrigées et certaines étiquettes erronées rectifiées.
- `text` : le contenu du tweet (variable explicative)
- `target` : **1** = le tweet parle d'une vraie catastrophe, **0** = non (variable cible)
- `keyword`, `location` : métadonnées (non utilisées ici, on se concentre sur le texte)

**Objectif de cette partie.**
1. Appliquer la chaîne NLP : prétraitement → tokenisation → vectorisation.
2. Implémenter **trois modèles**, en nous inspirant directement des codes d'exemple des TP 2.1 et 2.2 :
   - **Modèle 1 – Naive Bayes multinomial + TF-IDF** (inspiré de `naive_bayes.py` et du TP 2.1)
   - **Modèle 2 – SVM linéaire + TF-IDF** (inspiré de `support_vector.py`)
   - **Modèle 3 – CNN 1D avec embeddings**, alimenté par l'algorithme `get_top_xwords` (inspiré de `deep_learning.py`)
3. Évaluer, comparer et interpréter les performances (bonus : comparaison de modèles).

## 1. Installation et imports

On travaille sur **Google Colab**. La plupart des librairies y sont déjà installées. On vérifie seulement `kagglehub`, qui sert à télécharger le dataset via l'API Kaggle.

*Aucun GPU n'est nécessaire : le dataset est petit (~7 600 tweets) et le CNN s'entraîne en moins d'une minute sur CPU.*

In [ ]:
!pip install -q kagglehub

In [ ]:
# --- Librairies générales ---
import os, re, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# --- NLP : NLTK pour la tokenisation, les stopwords et la lemmatisation ---
import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)   # ressources linguistiques nécessaires
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize

# --- Machine Learning : scikit-learn ---
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

# --- Deep Learning : Keras 3 (via TensorFlow) ---
import keras
from keras import layers

# Graine aléatoire fixée partout -> résultats reproductibles
# (absent des scripts du TP 2.2, où chaque exécution donnait un score différent)
SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

sns.set_theme(style='whitegrid')
print('Keras', keras.__version__)

## 2. Chargement du jeu de données via l'API Kaggle

On utilise `kagglehub`, la librairie officielle de Kaggle. Pour un dataset public, elle fonctionne sur Colab sans configuration.

*Si Kaggle demande une authentification, il faut ajouter `KAGGLE_USERNAME` et `KAGGLE_KEY` dans les **Secrets** de Colab (icône 🔑 à gauche). On les trouve dans Kaggle → Settings → API → "Create New Token".*

Le dataset contient un fichier *train* (étiqueté) et un fichier *test* (**non étiqueté**, car c'était le test de la compétition). Pour évaluer nos modèles, on a besoin des vraies étiquettes. **On travaille donc uniquement sur le fichier train**, qu'on découpera nous-mêmes en entraînement et test.

In [ ]:
import kagglehub

# Téléchargement (mis en cache : un second appel ne retélécharge pas)
path = kagglehub.dataset_download("vbmokin/nlp-with-disaster-tweets-cleaning-data")
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
print('Fichiers disponibles :')
for f in csv_files:
    print('  -', os.path.basename(f))

# On sélectionne automatiquement le fichier d'entraînement (celui qui contient la cible)
train_file = [f for f in csv_files if 'train' in os.path.basename(f).lower()][0]
df = pd.read_csv(train_file)
print(f'\nFichier utilisé : {os.path.basename(train_file)} -> {df.shape[0]} tweets, {df.shape[1]} colonnes')
df.head()

💬 **Commentaire.** Chaque ligne est un tweet. Seules les colonnes `text` (entrée) et `target` (sortie à prédire) sont utilisées. Le problème est une **classification binaire supervisée**.

## 3. Vérifications rapides avant modélisation

L'exploration détaillée a été faite dans la partie précédente. Ici, on vérifie seulement ce qui impacte directement la modélisation et l'évaluation :
- des **textes manquants** empêcheraient la vectorisation ;
- les **doublons** sont dangereux : si le même tweet se retrouve dans le train et dans le test, le modèle "reconnaît" le tweet au lieu de généraliser, et le score est artificiellement gonflé (**fuite de données**) ;
- le **déséquilibre des classes** détermine quelles métriques utiliser.

In [ ]:
# 1) Textes manquants
print('Textes manquants :', df['text'].isna().sum())
df = df.dropna(subset=['text']).reset_index(drop=True)

# 2) Doublons : même texte présent plusieurs fois
n_dup = df.duplicated(subset='text').sum()
# Cas encore plus problématique : même texte mais étiquettes différentes (contradiction)
conflicts = df.groupby('text')['target'].nunique()
n_conflict_texts = (conflicts > 1).sum()
print(f'Doublons exacts sur le texte : {n_dup}')
print(f'Textes avec étiquettes contradictoires : {n_conflict_texts}')

# On retire les textes contradictoires (impossible de savoir quelle étiquette est juste)
# puis on garde une seule occurrence des autres doublons
df = df[~df['text'].isin(conflicts[conflicts > 1].index)]
df = df.drop_duplicates(subset='text').reset_index(drop=True)
print(f'Taille après dédoublonnage : {len(df)} tweets')

# 3) Répartition des classes
counts = df['target'].value_counts().sort_index()
print('\nRépartition des classes :')
print(pd.DataFrame({'effectif': counts, 'proportion (%)': (counts / len(df) * 100).round(1)}))

plt.figure(figsize=(5, 3.5))
sns.barplot(x=['0 - Pas une catastrophe', '1 - Catastrophe'], y=counts.values, palette=['#4C72B0', '#DD8452'])
plt.title('Répartition des classes')
plt.ylabel('Nombre de tweets')
plt.show()

💬 **Interprétation.** Les classes sont **légèrement déséquilibrées** (environ 57 % / 43 %). Un modèle naïf qui répond toujours "0" obtiendrait déjà ~57 % d'accuracy. Pour cette raison :
- on ne jugera **pas** les modèles sur l'accuracy seule, mais surtout sur le **F1-score**, la **précision** et le **rappel** de la classe 1 (catastrophe) ;
- on utilisera un **split stratifié** pour conserver ces proportions dans le train et le test ;
- on comparera nos modèles à une **baseline naïve** (DummyClassifier) pour prouver qu'ils apprennent réellement quelque chose.

## 4. Prétraitement NLP : nettoyage, tokenisation, lemmatisation

Le dataset est déjà "pré-nettoyé" par son auteur (URLs, contractions, fautes). Il faut quand même **normaliser** le texte pour les modèles. On reprend la logique du TP 2.1 (regex + minuscules + stopwords) en corrigeant ses défauts :

| Étape | Pourquoi |
|---|---|
| Minuscules | "Fire", "FIRE" et "fire" doivent être le même mot |
| `n't` → ` not` | Dans le TP 2.1, "wasn't" devenait "wasn t", puis "wasn" était supprimé comme stopword : **la négation était perdue**. On la préserve ici. |
| Suppression URLs, @mentions, `&amp;` | Bruit propre à Twitter, sans valeur sémantique |
| `#` retiré mais mot conservé | `#earthquake` → `earthquake` : le hashtag est souvent le mot le plus informatif |
| Garder uniquement les lettres | Supprime chiffres, ponctuation, emojis |
| **Tokenisation** `word_tokenize` (NLTK) | Découpe la phrase en mots (tokens). C'est la même fonction qu'utilise `get_top_xwords`, pour rester cohérent. |
| Stopwords **sauf négations** | "the", "is", "a" n'apportent rien. "no", "not", "never" peuvent inverser le sens. |
| **Lemmatisation** (au lieu du stemming du TP 2.1) | "floods"→"flood", "buildings"→"building". Le lemme reste un vrai mot, contrairement au stem de Porter ("disaster"→"disast"). Les mots importants restent donc lisibles lors de l'interprétation. |

In [ ]:
# Négations à conserver : on les retire de la liste des stopwords
NEGATIONS = {'no', 'not', 'nor', 'never', 'none', 'nobody', 'nothing', 'neither', 'nowhere'}
STOP_WORDS = set(stopwords.words('english')) - NEGATIONS   # set() -> recherche O(1)
lemmatizer = WordNetLemmatizer()
# NB : dans le TP 2.1, stopwords et PorterStemmer étaient recréés à chaque itération de la boucle.
# Ici on les crée une seule fois, en dehors de la fonction.

def preprocess(text):
    # Transforme un tweet brut en texte normalisé (tokens lemmatisés séparés par des espaces)
    text = str(text).lower()                              # 1. minuscules
    text = re.sub(r"won't", 'will not', text)              # 2. négations irrégulières
    text = re.sub(r"can't", 'can not', text)
    text = re.sub(r"n't\b", ' not', text)                  #    don't -> do not, wasn't -> was not
    text = re.sub(r'http\S+|www\.\S+', ' ', text)          # 3. URLs
    text = re.sub(r'@\w+', ' ', text)                      # 4. mentions @user
    text = re.sub(r'&amp;', ' and ', text)                 # 5. entités HTML
    text = text.replace('#', ' ')                          # 6. on garde le mot du hashtag
    text = re.sub(r'[^a-z\s]', ' ', text)                  # 7. uniquement des lettres
    tokens = word_tokenize(text)                           # 8. TOKENISATION
    tokens = [lemmatizer.lemmatize(t) for t in tokens      # 9. lemmatisation
              if t not in STOP_WORDS and len(t) > 1]       #    + stopwords + lettres isolées
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)

# Vérification visuelle avant / après
pd.set_option('display.max_colwidth', 120)
df[['text', 'clean_text', 'target']].sample(8, random_state=SEED)

In [ ]:
# Tweets devenus vides après nettoyage (ex. un tweet composé uniquement d'un lien) -> inutilisables
empty = (df['clean_text'].str.strip() == '').sum()
print(f'Tweets vides après prétraitement : {empty}')
df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)

# Longueur des tweets en nombre de tokens : utile pour paramétrer le CNN plus tard
df['n_tokens'] = df['clean_text'].str.split().str.len()
print(df['n_tokens'].describe().round(1))

💬 **Commentaire.** Après nettoyage, un tweet contient en moyenne une dizaine de tokens. Ce sont des textes **très courts**, ce qui aura deux conséquences :
- la matrice TF-IDF sera très **creuse** (chaque tweet n'active qu'une dizaine de colonnes sur plusieurs milliers) ;
- pour le CNN, une longueur de séquence d'une vingtaine de tokens suffira. Les 150 du TP 2.2 étaient adaptés à des critiques de vin, bien plus longues.

## 5. Séparation entraînement / test

⚠️ **Point méthodologique clé.** Dans les scripts du TP 2.2 (`naive_bayes.py`, `support_vector.py`), le TF-IDF était calculé sur **tout** le corpus *avant* le `train_test_split`. Le vocabulaire et les pondérations IDF "voyaient" donc le jeu de test : c'est une **fuite de données**, qui rend le score légèrement optimiste.

Ici, on sépare les données **en premier**. Toute transformation (TF-IDF, vocabulaire top-x du CNN) sera apprise **uniquement sur le train**, puis appliquée au test.

- 80 % entraînement / 20 % test (comme dans le TP 2.1)
- `stratify=y` : les proportions 57/43 sont conservées dans les deux jeux
- `random_state` fixé : même découpage à chaque exécution

In [ ]:
X = df['clean_text']
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

print(f'Train : {len(X_train)} tweets  |  proportion de catastrophes = {y_train.mean():.3f}')
print(f'Test  : {len(X_test)} tweets  |  proportion de catastrophes = {y_test.mean():.3f}')

💬 Les proportions de la classe 1 sont quasi identiques dans les deux jeux : la stratification a bien fonctionné. Le jeu de test (~1 500 tweets) ne sera **utilisé qu'une seule fois**, pour l'évaluation finale. Le réglage des hyperparamètres se fera par validation croisée sur le train.

## 6. Outils d'évaluation communs

Pour comparer les modèles équitablement, on définit **une seule fonction d'évaluation**, appliquée à tous. Les scripts d'exemple calculaient l'accuracy avec une boucle manuelle. On utilise ici les métriques de scikit-learn, plus complètes :

| Métrique | Question à laquelle elle répond |
|---|---|
| **Accuracy** | Quelle proportion de tweets est bien classée ? |
| **Précision (classe 1)** | Quand le modèle déclenche une alerte, a-t-il raison ? (peu de fausses alertes) |
| **Rappel (classe 1)** | Parmi les vraies catastrophes, combien sont détectées ? (peu de catastrophes manquées) |
| **F1-score** | Moyenne harmonique précision/rappel. C'est la **métrique principale** ici, et c'était aussi celle de la compétition Kaggle. |
| **ROC-AUC** | Capacité à bien *classer* les tweets par score de risque, indépendamment du seuil de décision |
| **Matrice de confusion** | Détail des erreurs : fausses alertes (FP) et catastrophes manquées (FN) |

*Dans notre cas d'usage, un **faux négatif** (catastrophe réelle non détectée) est plus grave qu'une fausse alerte : le **rappel** mérite une attention particulière.*

In [ ]:
results = []   # on stocke les métriques de chaque modèle pour la comparaison finale
LABELS = ['0 - Pas catastrophe', '1 - Catastrophe']

def evaluate_model(name, y_true, y_pred, y_score, fit_time=None):
    # Calcule et affiche toutes les métriques d'un modèle, trace sa matrice de confusion,
    # et enregistre les résultats dans la liste `results`.
    # y_score = score continu (probabilité ou distance à l'hyperplan), utilisé pour la ROC-AUC
    metrics = {
        'Modèle': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Précision': precision_score(y_true, y_pred),
        'Rappel': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_score),
        "Temps d'entraînement (s)": fit_time,
    }
    print(f'=== {name} ===')
    print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))

    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100    # % par ligne (par vraie classe)
    annot = np.array([[f'{cm[i, j]}\n({cm_pct[i, j]:.1f}%)' for j in range(2)] for i in range(2)])
    plt.figure(figsize=(4.5, 3.8))
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', cbar=False,
                xticklabels=['Prédit 0', 'Prédit 1'], yticklabels=['Réel 0', 'Réel 1'])
    plt.title(f'Matrice de confusion – {name}')
    plt.show()

    tn, fp, fn, tp = cm.ravel()
    print(f'-> Fausses alertes (FP) : {fp}  |  Catastrophes manquées (FN) : {fn}')
    results.append({**metrics, 'y_score': y_score, 'y_pred': y_pred})
    return metrics

# Validation croisée stratifiée à 5 plis, utilisée pour régler les hyperparamètres
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

### 6.1 Baseline naïve

Avant tout modèle, on mesure le score d'un classifieur qui prédit **toujours la classe majoritaire**. Tout modèle utile doit faire nettement mieux.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
print(f'Accuracy baseline  : {accuracy_score(y_test, y_pred_dummy):.3f}')
print(f'F1 baseline        : {f1_score(y_test, y_pred_dummy, zero_division=0):.3f}')

💬 La baseline atteint ~57 % d'accuracy **sans rien comprendre** au texte, et un F1 de **0**, car elle ne détecte aucune catastrophe. Cela confirme qu'il faut juger les modèles sur le F1 et non sur l'accuracy.

## 7. Vectorisation : TF-IDF

Les modèles ML ne comprennent que des nombres. Il faut donc transformer chaque tweet en vecteur.

- Le **Bag of Words** (`CountVectorizer`, TP 2.1) compte les occurrences de chaque mot.
- Le **TF-IDF** (TP 2.2) pondère ce compte : **TF** (fréquence du mot dans le tweet) × **IDF** (log inverse de la proportion de tweets contenant ce mot). Un mot présent partout ("people", "like") reçoit un poids faible. Un mot rare et spécifique ("wildfire", "evacuation") reçoit un poids fort.

**Paramètres choisis :**
- `ngram_range=(1, 2)` : unigrammes **et bigrammes**. Le bigramme "forest fire" ou "not safe" porte plus de sens que ses mots isolés. Les exemples du TP n'utilisaient que des unigrammes. La validation croisée décidera si les bigrammes aident vraiment.
- `min_df=2` : on ignore les mots vus dans un seul tweet (fautes, noms rares). Cela réduit le bruit et la dimension.
- `sublinear_tf=True` : on utilise log(1+tf), pour qu'un mot répété 5 fois ne pèse pas 5 fois plus.

On intègre le TF-IDF **dans un `Pipeline`** avec le classifieur. Ainsi, lors de la validation croisée, le vocabulaire est réappris sur chaque pli d'entraînement : aucune fuite de données possible.

In [ ]:
# Aperçu de la représentation TF-IDF (uniquement pour illustrer ; les modèles utilisent le Pipeline)
tfidf_demo = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train_tfidf = tfidf_demo.fit_transform(X_train)     # fit sur le train uniquement
X_test_tfidf = tfidf_demo.transform(X_test)           # simple transformation du test

print(f'Dimension de la matrice TF-IDF (train) : {X_train_tfidf.shape}')
print(f'Taux de remplissage : {X_train_tfidf.nnz / np.prod(X_train_tfidf.shape) * 100:.3f} %')
print('Exemples de termes du vocabulaire :', list(tfidf_demo.get_feature_names_out()[::800][:10]))

💬 **Interprétation.** Chaque tweet devient un vecteur de plusieurs milliers de dimensions, mais moins de 0,5 % des cases sont non nulles : la matrice est **très creuse**. C'est précisément le type de données où Naive Bayes et les SVM linéaires excellent. Ils restent rapides et robustes en haute dimension, même avec peu d'exemples.

## 8. Modèle 1 : Naive Bayes multinomial (TF-IDF)

**Principe.** Naive Bayes applique le théorème de Bayes : il calcule P(catastrophe | mots du tweet) à partir de la fréquence de chaque mot dans les tweets de chaque classe. Il fait l'hypothèse "naïve" que les mots sont indépendants entre eux. C'est faux en réalité, mais cela marche très bien en pratique pour le texte.

**Pourquoi Multinomial et non Gaussien (TP 2.1) ?** Le `GaussianNB` suppose que chaque variable suit une loi normale continue. Des comptes ou poids TF-IDF (majoritairement 0, positifs, creux) n'ont rien de gaussien. `MultinomialNB` modélise justement des fréquences de mots : c'est la variante théoriquement adaptée au texte, et celle utilisée dans `naive_bayes.py`.

**Justification du choix.** C'est le modèle de référence historique en classification de texte (détection de spam). Il est très rapide, fonctionne bien sur de petits datasets et reste **interprétable** : on peut lire quels mots poussent vers "catastrophe".

**Hyperparamètre réglé :** `alpha` (lissage de Laplace). Il évite qu'un mot jamais vu dans une classe donne une probabilité nulle. Plus alpha est grand, plus le modèle est "prudent".

In [ ]:
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, sublinear_tf=True)),
    ('clf', MultinomialNB()),
])

# Grille de recherche : lissage alpha x unigrammes seuls / unigrammes + bigrammes
nb_grid = GridSearchCV(
    nb_pipeline,
    param_grid={'clf__alpha': [0.1, 0.3, 0.5, 1.0],
                'tfidf__ngram_range': [(1, 1), (1, 2)]},
    cv=cv, scoring='f1', n_jobs=-1)

t0 = time.time()
nb_grid.fit(X_train, y_train)          # validation croisée 5 plis sur le TRAIN uniquement
nb_time = time.time() - t0

print('Meilleurs hyperparamètres :', nb_grid.best_params_)
cv_nb = pd.DataFrame(nb_grid.cv_results_)
best = cv_nb.loc[nb_grid.best_index_]
print(f"F1 en validation croisée : {best['mean_test_score']:.3f} ± {best['std_test_score']:.3f}")

💬 **Commentaire.** La validation croisée entraîne et évalue le modèle 5 fois sur des sous-parties différentes du train, pour chacune des 8 combinaisons de la grille. L'écart-type indique la **stabilité** du modèle : un écart faible (≈ 0,01) signifie que la performance ne dépend pas du hasard du découpage. Le modèle final (`best_estimator_`) est automatiquement réentraîné sur tout le train avec les meilleurs paramètres.

In [ ]:
nb_model = nb_grid.best_estimator_
y_pred_nb = nb_model.predict(X_test)
y_score_nb = nb_model.predict_proba(X_test)[:, 1]    # probabilité d'être une catastrophe
evaluate_model('Naive Bayes (TF-IDF)', y_test, y_pred_nb, y_score_nb, nb_time)

💬 **Lecture des résultats.** Comparer le F1 du test au F1 de validation croisée : s'ils sont proches, le modèle **généralise** bien et n'est pas sur-ajusté. Dans la matrice de confusion, la case *Réel 1 / Prédit 0* compte les catastrophes manquées. Naive Bayes a tendance à favoriser la classe majoritaire, via la probabilité a priori P(classe) : on s'attend donc à une précision plutôt élevée et un rappel un peu plus faible.

## 9. Modèle 2 : SVM linéaire (TF-IDF)

**Principe.** Le SVM (Support Vector Machine) cherche l'**hyperplan** qui sépare le mieux les deux classes dans l'espace TF-IDF, en maximisant la **marge**, c'est-à-dire la distance entre l'hyperplan et les points les plus proches (les vecteurs de support). Contrairement à Naive Bayes, il ne suppose pas l'indépendance des mots : il apprend directement une frontière de décision discriminante.

**Pourquoi linéaire ?** En texte, les données sont en très haute dimension (des milliers de mots). Dans un tel espace, les classes sont souvent **déjà linéairement séparables** : un noyau non linéaire (RBF) coûte bien plus cher sans gain notable. `support_vector.py` utilisait `SVC(kernel='linear')`. Nous utilisons `LinearSVC`, qui résout exactement le même problème mais avec un algorithme optimisé pour les matrices creuses, beaucoup plus rapide.

**Hyperparamètres réglés :**
- `C` : compromis marge / erreurs. Un C petit donne une marge large, un modèle plus simple et une meilleure généralisation. Un C grand colle davantage aux données d'entraînement (risque de sur-apprentissage).
- `class_weight='balanced'` : donne plus de poids à la classe minoritaire (catastrophes), ce qui peut **augmenter le rappel**.

In [ ]:
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, sublinear_tf=True)),
    ('clf', LinearSVC(random_state=SEED, max_iter=5000)),
])

svm_grid = GridSearchCV(
    svm_pipeline,
    param_grid={'clf__C': [0.05, 0.1, 0.3, 1.0],
                'clf__class_weight': [None, 'balanced'],
                'tfidf__ngram_range': [(1, 1), (1, 2)]},
    cv=cv, scoring='f1', n_jobs=-1)

t0 = time.time()
svm_grid.fit(X_train, y_train)
svm_time = time.time() - t0

print('Meilleurs hyperparamètres :', svm_grid.best_params_)
cv_svm = pd.DataFrame(svm_grid.cv_results_)
best = cv_svm.loc[svm_grid.best_index_]
print(f"F1 en validation croisée : {best['mean_test_score']:.3f} ± {best['std_test_score']:.3f}")

In [ ]:
# Visualisation de l'effet de C sur le F1 (avec les autres paramètres fixés à leur meilleure valeur)
bp = svm_grid.best_params_
mask = ((cv_svm['param_clf__class_weight'].astype(str) == str(bp['clf__class_weight'])) &
        (cv_svm['param_tfidf__ngram_range'].astype(str) == str(bp['tfidf__ngram_range'])))
sub = cv_svm[mask].sort_values('param_clf__C')
plt.figure(figsize=(5.5, 3.5))
plt.errorbar(sub['param_clf__C'].astype(float), sub['mean_test_score'], yerr=sub['std_test_score'],
             marker='o', capsize=4)
plt.xscale('log'); plt.xlabel('C (échelle log)'); plt.ylabel('F1 moyen (CV 5 plis)')
plt.title('SVM : influence de la régularisation C')
plt.show()

💬 **Interprétation de la courbe.** Pour un C trop petit, le modèle est trop contraint (sous-apprentissage) et le F1 baisse. Pour un C trop grand, il mémorise le bruit du train (sur-apprentissage) et le F1 en validation baisse à nouveau. Le meilleur C se situe entre les deux. Sur des tweets courts et bruités, on s'attend à un **C plutôt faible** (forte régularisation).

In [ ]:
svm_model = svm_grid.best_estimator_
y_pred_svm = svm_model.predict(X_test)
# LinearSVC ne donne pas de probabilité : on utilise la distance signée à l'hyperplan comme score
y_score_svm = svm_model.decision_function(X_test)
evaluate_model('SVM linéaire (TF-IDF)', y_test, y_pred_svm, y_score_svm, svm_time)

## 10. Modèle 3 : CNN 1D avec embeddings (Deep Learning)

### 10.1 Vectorisation par indices avec l'algorithme `get_top_xwords`

Un réseau de neurones avec couche d'**Embedding** ne prend pas un vecteur TF-IDF en entrée, mais une **séquence d'indices de mots** qui respecte l'ordre des mots. On reprend l'algorithme `get_top_xwords` du TP 2.2 (S. Wang) :
1. `count_top_x_words` : compte la fréquence de chaque token (`word_tokenize`) et garde les `top_x` mots les plus fréquents (en sautant éventuellement les `skip_top_n` premiers) ;
2. `replace_top_x_words_with_vectors` : remplace chaque mot d'un tweet par son indice dans ce vocabulaire. Les mots hors vocabulaire sont ignorés ;
3. `filter_to_top_x` : enchaîne les deux.

On écrit le module dans `lib/get_top_xwords.py`, exactement comme dans l'arborescence du projet d'origine, pour pouvoir faire le même `import`.

In [ ]:
os.makedirs('lib', exist_ok=True)
open('lib/__init__.py', 'w').close()     # fait de `lib` un package Python importable

In [ ]:
%%writefile lib/get_top_xwords.py
from nltk import word_tokenize
from collections import defaultdict


def count_top_x_words(corpus, top_x, skip_top_n):
    count = defaultdict(lambda: 0)
    for c in corpus:
        for w in word_tokenize(c):
            count[w] += 1
    count_tuples = sorted([(w, c) for w, c in count.items()], key=lambda x: x[1], reverse=True)
    return [i[0] for i in count_tuples[skip_top_n: skip_top_n + top_x]]


def replace_top_x_words_with_vectors(corpus, top_x):
    topx_dict = {top_x[i]: i for i in range(len(top_x))}

    return [
        [topx_dict[w] for w in word_tokenize(s) if w in topx_dict]
        for s in corpus
    ], topx_dict


def filter_to_top_x(corpus, n_top, skip_n_top=0):
    top_x = count_top_x_words(corpus, n_top, skip_n_top)
    return replace_top_x_words_with_vectors(corpus, top_x)

**Deux adaptations nécessaires par rapport à l'utilisation dans `deep_learning.py` :**

1. **Éviter la fuite de données.** `deep_learning.py` appelait `filter_to_top_x` sur tout le corpus, avant le split : le vocabulaire était construit en voyant le test. Ici, on appelle `count_top_x_words` **sur le train seulement**, puis `replace_top_x_words_with_vectors` sur le train et le test avec ce même vocabulaire. On utilise les fonctions du module telles quelles, simplement dans le bon ordre.

2. **Corriger le conflit avec le padding.** Le module attribue l'indice **0** au mot le plus fréquent. Or `pad_sequences` complète les tweets courts avec des **0**. Le réseau ne pouvait donc pas distinguer "mot le plus fréquent" et "case vide". On décale tous les indices de +1 : **0 = padding** et les mots vont de 1 à N. La couche Embedding a donc une taille N + 1.

**`skip_top_n`** : `deep_learning.py` sautait les 10 mots les plus fréquents, parce qu'il n'avait pas retiré les stopwords et que ceux-ci dominaient le classement. Nos stopwords sont **déjà supprimés** : les mots les plus fréquents sont maintenant informatifs ("fire", "news", "people"…), on ne saute donc rien (`SKIP_TOP = 0`).

In [ ]:
from lib.get_top_xwords import count_top_x_words, replace_top_x_words_with_vectors

N_TOP = 5000    # taille du vocabulaire conservé (2500 dans le TP 2.2 ; les tweets ont un vocabulaire plus varié)
SKIP_TOP = 0    # les stopwords sont déjà retirés -> aucun mot fréquent à ignorer

# 1) Vocabulaire appris sur le TRAIN uniquement
top_words = count_top_x_words(list(X_train), N_TOP, SKIP_TOP)

# 2) Conversion en séquences d'indices avec CE vocabulaire, pour le train et le test
train_seq, word_index = replace_top_x_words_with_vectors(list(X_train), top_words)
test_seq, _ = replace_top_x_words_with_vectors(list(X_test), top_words)

# 3) Décalage +1 : l'indice 0 est réservé au padding
train_seq = [[i + 1 for i in s] for s in train_seq]
test_seq = [[i + 1 for i in s] for s in test_seq]
VOCAB_SIZE = len(top_words) + 1

print(f'Taille du vocabulaire : {len(top_words)} mots (+1 pour le padding)')
print('10 mots les plus fréquents :', top_words[:10])
print('\nExemple :')
print('  tweet   :', X_train.iloc[0])
print('  indices :', train_seq[0])

# Couverture : part des tokens du test présents dans le vocabulaire appris sur le train
n_tok_test = sum(len(word_tokenize(s)) for s in X_test)
print(f'\nCouverture du vocabulaire sur le test : {sum(len(s) for s in test_seq) / n_tok_test * 100:.1f} % des tokens')

💬 **Interprétation.** La couverture indique la part des mots du test que le CNN "connaît". Les mots restants (hors vocabulaire) sont simplement ignorés. Une couverture autour de 85–90 % est satisfaisante : les mots perdus sont surtout rares (noms propres, fautes), donc peu informatifs.

### 10.2 Padding : des séquences de même longueur

Un réseau de neurones attend des entrées de **taille fixe**. `pad_sequences` complète les tweets courts avec des 0 (au début, par défaut) et tronque les tweets trop longs. On choisit la longueur à partir des données : le **99ᵉ percentile** de la longueur des tweets du train. Presque aucun tweet n'est tronqué, sans gaspiller de calcul.

In [ ]:
lengths = [len(s) for s in train_seq]
MAX_LEN = int(np.percentile(lengths, 99))
print(f'Longueur max retenue : {MAX_LEN} tokens (le TP 2.2 utilisait 150 pour des critiques de vin)')

# keras.utils.pad_sequences remplace keras.preprocessing.sequence, supprimé dans Keras 3
X_train_pad = keras.utils.pad_sequences(train_seq, maxlen=MAX_LEN)
X_test_pad = keras.utils.pad_sequences(test_seq, maxlen=MAX_LEN)
print('Forme des tenseurs :', X_train_pad.shape, X_test_pad.shape)
print('Exemple après padding :', X_train_pad[0])

# Jeu de validation (10 % du train, stratifié) pour surveiller le sur-apprentissage pendant l'entraînement.
# Le jeu de test reste intouché jusqu'à l'évaluation finale.
X_tr_pad, X_val_pad, y_tr, y_val = train_test_split(
    X_train_pad, y_train, test_size=0.10, stratify=y_train, random_state=SEED)

### 10.3 Architecture du CNN

**Principe.**
- **Embedding** : chaque indice de mot devient un vecteur dense de 64 dimensions, **appris** pendant l'entraînement. Des mots au sens proche ("fire", "blaze", "wildfire") finissent avec des vecteurs proches. C'est la technique de *word embedding* décrite dans l'énoncé (famille Word2Vec/GloVe), apprise ici directement sur notre tâche.
- **Conv1D** : des filtres glissent sur la séquence de mots et détectent des **motifs locaux de 3 mots consécutifs** (des "n-grammes appris"), par exemple "forest fire near" ou "building collapse kill". C'est l'intérêt du CNN : il **tient compte de l'ordre des mots**, contrairement au TF-IDF.
- **Dense + sigmoïde** : décision finale, une probabilité entre 0 et 1.

**Modifications par rapport à `deep_learning.py`, et pourquoi :**

| TP 2.2 (vins) | Ici (tweets) | Raison |
|---|---|---|
| `Dense(10, softmax)` + `categorical_crossentropy` | `Dense(1, sigmoid)` + `binary_crossentropy` | Problème binaire et non à 10 classes |
| `Conv1D(50, 5)` | `Conv1D(64, 3, relu)` | Tweets courts (~10 mots) : des fenêtres de 3 mots sont plus adaptées. L'activation ReLU manquait dans l'original. |
| `Flatten()` | `GlobalMaxPooling1D()` | Garde, pour chaque filtre, sa détection la plus forte **où qu'elle soit** dans le tweet. Beaucoup moins de paramètres, donc moins de sur-apprentissage. |
| — | `Dropout(0.5)` | Désactive aléatoirement 50 % des neurones à l'entraînement : régularisation indispensable avec ~6 000 exemples |
| `epochs=3` fixe | `EarlyStopping` | On arrête automatiquement quand la perte de validation remonte, et on garde les meilleurs poids |
| `input_length=...` | `keras.Input(shape=...)` | Syntaxe Keras 3 : l'ancien code ne s'exécute plus sur les versions actuelles |

In [ ]:
keras.utils.set_random_seed(SEED)

EMBEDDING_DIM = 64      # même dimension que dans deep_learning.py

cnn = keras.Sequential([
    keras.Input(shape=(MAX_LEN,)),
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    layers.GlobalMaxPooling1D(),
    layers.Dropout(0.5),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),
], name='cnn_disaster_tweets')

cnn.compile(loss='binary_crossentropy', optimizer='adam',
            metrics=['accuracy', keras.metrics.AUC(name='auc')])
cnn.summary()

💬 **Commentaire.** L'essentiel des paramètres se trouve dans la couche Embedding (5 001 mots × 64 dimensions ≈ 320 000). C'est beaucoup au regard de ~6 000 tweets d'entraînement, d'où l'importance du Dropout et de l'arrêt anticipé pour limiter le sur-apprentissage.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

t0 = time.time()
history = cnn.fit(X_tr_pad, y_tr,
                  validation_data=(X_val_pad, y_val),
                  epochs=20, batch_size=64,          # batch_size identique au TP 2.2
                  callbacks=[early_stop], verbose=2)
cnn_time = time.time() - t0
print(f'\nArrêt après {len(history.history["loss"])} époques '
      f'(meilleure époque : {int(np.argmin(history.history["val_loss"])) + 1})')

In [ ]:
# Courbes d'apprentissage : diagnostic du sur-apprentissage
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for k, (metric, title) in enumerate([('loss', 'Perte (binary cross-entropy)'), ('accuracy', 'Accuracy')]):
    ax[k].plot(history.history[metric], 'o-', label='Entraînement')
    ax[k].plot(history.history['val_' + metric], 'o-', label='Validation')
    ax[k].set_title(title); ax[k].set_xlabel('Époque'); ax[k].legend()
plt.tight_layout(); plt.show()

💬 **Interprétation des courbes.** La perte d'entraînement diminue continuellement : le réseau apprend. La perte de **validation**, elle, atteint un minimum après quelques époques puis **remonte** : c'est le signe du **sur-apprentissage**, le réseau commence à mémoriser le train. L'`EarlyStopping` restaure automatiquement les poids de l'époque au minimum de validation. On voit aussi pourquoi les 3 époques fixes du TP 2.2 étaient arbitraires : le bon nombre dépend des données.

In [ ]:
y_score_cnn = cnn.predict(X_test_pad, verbose=0).ravel()   # probabilité de catastrophe
y_pred_cnn = (y_score_cnn >= 0.5).astype(int)              # seuil de décision à 0,5
evaluate_model('CNN 1D (embeddings)', y_test, y_pred_cnn, y_score_cnn, cnn_time)

## 11. Comparaison des modèles (bonus)

On compare les trois modèles sur **le même jeu de test**, avec les mêmes métriques.

In [ ]:
res_df = (pd.DataFrame(results)
          .drop(columns=['y_score', 'y_pred'])
          .set_index('Modèle')
          .sort_values('F1', ascending=False))
display(res_df.style.format('{:.3f}').highlight_max(
    subset=['Accuracy', 'Précision', 'Rappel', 'F1', 'ROC-AUC'], color='#c6efce'))

# Graphique comparatif
res_df[['Accuracy', 'Précision', 'Rappel', 'F1', 'ROC-AUC']].plot.bar(figsize=(10, 4), rot=0, width=0.8)
plt.ylim(0.5, 1); plt.title('Comparaison des performances sur le jeu de test')
plt.legend(loc='lower right', ncol=5, fontsize=8); plt.ylabel('Score')
plt.show()

In [ ]:
# Courbes ROC superposées
plt.figure(figsize=(5.5, 5))
for r in results:
    fpr, tpr, _ = roc_curve(y_test, r['y_score'])
    plt.plot(fpr, tpr, label=f"{r['Modèle']} (AUC = {r['ROC-AUC']:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Hasard (AUC = 0.5)')
plt.xlabel('Taux de faux positifs'); plt.ylabel('Taux de vrais positifs (rappel)')
plt.title('Courbes ROC'); plt.legend(fontsize=8)
plt.show()

In [ ]:
# Synthèse automatique : ce texte s'adapte aux résultats obtenus lors de l'exécution
best_name = res_df['F1'].idxmax()
print(f"Meilleur modèle selon le F1 : {best_name} (F1 = {res_df.loc[best_name, 'F1']:.3f})")
print(f"Meilleur rappel (moins de catastrophes manquées) : {res_df['Rappel'].idxmax()} "
      f"({res_df['Rappel'].max():.3f})")
print(f"Meilleure précision (moins de fausses alertes) : {res_df['Précision'].idxmax()} "
      f"({res_df['Précision'].max():.3f})")
fastest = res_df["Temps d'entraînement (s)"].idxmin()
print(f"Modèle le plus rapide à entraîner : {fastest}")
gap = res_df['F1'].max() - res_df['F1'].min()
print(f"Écart de F1 entre le meilleur et le moins bon modèle : {gap:.3f} "
      f"({'faible : les modèles se valent' if gap < 0.02 else 'significatif'})")

💬 **Interprétation comparative.**

*(À affiner selon les chiffres obtenus ci-dessus.)*

- **Les trois modèles battent largement la baseline** (F1 = 0) : ils ont bien appris à reconnaître les tweets de catastrophe à partir de leur contenu.
- **Modèles linéaires vs Deep Learning.** Sur ce jeu de données, on s'attend à ce que les modèles TF-IDF (surtout le SVM) soient **au niveau du CNN, voire meilleurs**. Trois raisons :
  1. **Peu de données** : ~6 000 tweets suffisent à un modèle linéaire, mais c'est peu pour apprendre de bons embeddings à partir de zéro (≈ 350 000 paramètres).
  2. **Textes très courts** : l'avantage du CNN, capter l'ordre des mots, compte peu quand un tweet ne fait qu'une dizaine de mots. Les bigrammes TF-IDF capturent déjà les expressions clés.
  3. **Signal lexical fort** : la présence de mots comme "wildfire", "evacuate" ou "killed" suffit souvent à décider, ce qui est exactement ce que capturent Naive Bayes et le SVM.
- **SVM vs Naive Bayes.** Le SVM est discriminant : il apprend directement la frontière et gère les mots corrélés. Naive Bayes, lui, compte chaque mot comme indépendant et "double-compte" des mots liés ("forest" + "fire"). Le SVM est donc généralement légèrement meilleur. Naive Bayes reste le plus rapide et un excellent point de départ.
- **Choix pour le cas d'usage.** Pour un service d'alerte, le modèle retenu est celui avec le meilleur **F1**. À F1 comparable, on préfère celui avec le meilleur **rappel**, car manquer une vraie catastrophe coûte plus cher qu'une fausse alerte. On tient compte aussi du coût : un modèle linéaire s'entraîne en quelques secondes et s'explique facilement, ce qui est un vrai avantage opérationnel.

## 12. Interprétabilité : quels mots déclenchent la détection ?

Un avantage des modèles linéaires est qu'on peut **lire ce qu'ils ont appris** :
- pour **Naive Bayes** : log P(mot | catastrophe) − log P(mot | non-catastrophe). Plus la valeur est grande, plus le mot est typique des catastrophes ;
- pour le **SVM** : le coefficient de chaque mot dans l'équation de l'hyperplan. Un coefficient positif pousse vers "catastrophe", un coefficient négatif vers "pas catastrophe".

In [ ]:
def top_features(names, scores, n=15):
    order = np.argsort(scores)
    return names[order[-n:]], scores[order[-n:]], names[order[:n]], scores[order[:n]]

nb_names = nb_model.named_steps['tfidf'].get_feature_names_out()
nb_clf = nb_model.named_steps['clf']
nb_scores = nb_clf.feature_log_prob_[1] - nb_clf.feature_log_prob_[0]

svm_names = svm_model.named_steps['tfidf'].get_feature_names_out()
svm_scores = svm_model.named_steps['clf'].coef_.ravel()

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for k, (name, names, scores) in enumerate([('Naive Bayes', nb_names, nb_scores),
                                           ('SVM', svm_names, svm_scores)]):
    pos_w, pos_s, neg_w, neg_s = top_features(names, scores)
    axes[2 * k].barh(pos_w, pos_s, color='#DD8452'); axes[2 * k].set_title(f'{name} → CATASTROPHE')
    axes[2 * k + 1].barh(neg_w, neg_s, color='#4C72B0'); axes[2 * k + 1].set_title(f'{name} → PAS catastrophe')
plt.tight_layout(); plt.show()

💬 **Interprétation.** Les termes associés aux catastrophes devraient être très concrets : types d'événements (*wildfire, earthquake, flood, bombing*), conséquences (*killed, evacuation, casualty*) et vocabulaire journalistique (*news, police, california*). Du côté "pas catastrophe", on attend des mots du quotidien et de l'émotion (*love, lol, new, video, body*). Ils révèlent les usages figurés : on "explose" de rire, un morceau est "fire".

Le fait que les deux modèles, construits sur des principes très différents (probabiliste vs géométrique), s'appuient sur des mots similaires **renforce la confiance** dans ce qu'ils ont appris. Si un mot surprenant apparaît (nom propre, artefact du nettoyage), c'est un indice de biais du dataset à signaler.

## 13. Analyse des erreurs

Les métriques disent *combien* le modèle se trompe. L'analyse d'erreurs dit **pourquoi**. On examine les tweets mal classés par le meilleur modèle.

In [ ]:
best_res = next(r for r in results if r['Modèle'] == best_name)
err = pd.DataFrame({'tweet original': df.loc[X_test.index, 'text'].values,
                    'réel': y_test, 'prédit': best_res['y_pred'],
                    'score': np.round(best_res['y_score'], 3)})

print(f'Modèle analysé : {best_name}\n')
print('FAUX POSITIFS (alerte déclenchée à tort) :')
display(err[(err['réel'] == 0) & (err['prédit'] == 1)].head(8))
print('FAUX NÉGATIFS (catastrophe manquée) :')
display(err[(err['réel'] == 1) & (err['prédit'] == 0)].head(8))

💬 **Interprétation.** Les erreurs se classent en général en trois familles :
1. **Langage figuré ou ironie** (faux positifs) : "this song is the bomb", "my phone just died". Le vocabulaire de catastrophe est là, mais pas le sens. Un modèle "sac de mots" a du mal à le détecter, car il faudrait comprendre le contexte global.
2. **Catastrophes décrites de manière indirecte** (faux négatifs) : tweets sans mot-clé explicite, ou utilisant un vocabulaire rare, absent du train.
3. **Étiquettes discutables** : certains tweets sont ambigus même pour un humain (un article sur une catastrophe passée compte-t-il ?). Cela fixe un plafond de performance, quel que soit le modèle.

**Pistes d'amélioration :** utiliser des embeddings pré-entraînés (GloVe Twitter), exploiter la colonne `keyword`, ou utiliser un Transformer pré-entraîné (BERT/DistilBERT) qui comprend le contexte et gère mieux le langage figuré.

## 14. Démonstration : prédiction sur de nouveaux tweets

On teste les trois modèles sur des tweets inventés, dont certains pièges volontaires (sens figuré). Le nouveau texte passe exactement par la **même chaîne de prétraitement** que les données d'entraînement.

In [ ]:
new_tweets = [
    'Forest fire near La Ronge Sask. Canada, residents asked to evacuate',
    'Magnitude 6.2 earthquake hits the coast, several buildings collapsed',
    'This new album is fire, the second track is the bomb!',
    'I was dying of laughter at the party last night lol',
    'Flash flood warning issued for the county, avoid the roads',
]

clean_new = [preprocess(t) for t in new_tweets]
seq_new, _ = replace_top_x_words_with_vectors(clean_new, top_words)
pad_new = keras.utils.pad_sequences([[i + 1 for i in s] for s in seq_new], maxlen=MAX_LEN)

demo = pd.DataFrame({
    'tweet': new_tweets,
    'Naive Bayes': nb_model.predict(clean_new),
    'SVM': svm_model.predict(clean_new),
    'CNN': (cnn.predict(pad_new, verbose=0).ravel() >= 0.5).astype(int),
    'P(catastrophe) NB': np.round(nb_model.predict_proba(clean_new)[:, 1], 2),
})
demo

💬 **Interprétation.** Les deux premiers et le dernier tweet sont de vraies alertes et devraient être prédits **1**. Les tweets 3 et 4 sont des pièges au sens figuré ("fire", "bomb", "dying") et devraient être prédits **0**. Si un modèle se trompe sur les pièges, cela illustre concrètement la limite des approches par mots-clés identifiée dans l'analyse d'erreurs.

## 15. Conclusion

- **Chaîne NLP mise en œuvre :** nettoyage adapté aux tweets avec préservation des négations → tokenisation `word_tokenize` → lemmatisation → deux vectorisations complémentaires : **TF-IDF** (uni/bigrammes) pour les modèles ML, et **séquences d'indices** via l'algorithme `get_top_xwords` + **embeddings appris** pour le CNN.
- **Rigueur méthodologique** apportée par rapport aux codes d'exemple : split *avant* toute vectorisation (pas de fuite de données), `Pipeline` + validation croisée stratifiée pour régler les hyperparamètres, `MultinomialNB` au lieu de `GaussianNB`, correction du conflit indice 0 / padding, code Keras compatible avec les versions actuelles, graines fixées.
- **Évaluation complète :** baseline naïve, précision/rappel/F1/ROC-AUC, matrices de confusion, courbes d'apprentissage, courbes ROC, interprétation des mots discriminants et analyse d'erreurs.
- **Résultat :** voir la synthèse de la section 11. Le meilleur compromis performance / coût / interprétabilité pour un système d'alerte est celui qui maximise le F1 avec un rappel élevé.